In [0]:
# ── Change only these variables ───────────────────────────────────────
catalog = "workspace"                  # or stage_ica_restricted
# stage     ="  # or stage_ica_unrestricted
retention = "interval 60 days"       # or interval 30 days for stage
# ─────────────────────────────────────────────────────────────────────

# Step 1: Dynamically fetch all schemas in the catalog
schemas = spark.sql(f"""
    SELECT schema_name
    FROM {catalog}.information_schema.schemata
    WHERE schema_name != 'information_schema'
""").collect()

print(f"Found {len(schemas)} schemas in {catalog}\n")

total_success, total_failed = 0, 0

# Step 2: Loop through each schema dynamically
for schema_row in schemas:
    schema = schema_row.schema_name
    print(f"\n── Schema: {schema} ──")

    tables = spark.sql(f"""
        SELECT table_name
        FROM {catalog}.information_schema.tables
        WHERE table_schema      = '{schema}'
          AND data_source_format = 'DELTA'
    """).collect()

    print(f"   Found {len(tables)} Delta tables")

    for table_row in tables:
        fqn = f"`{catalog}`.`{schema}`.`{table_row.table_name}`"
        try:
            spark.sql(f"""
                ALTER TABLE {fqn} SET TBLPROPERTIES (
                    'delta.logRetentionDuration'         = '{retention}',
                    'delta.deletedFileRetentionDuration' = '{retention}'
                )
            """)
            print(f"   [OK]  {fqn}")
            total_success += 1
        except Exception as e:
            print(f"   [ERR] {fqn} → {e}")
            total_failed += 1

print(f"\n{'='*60}")
print(f"Done — Success: {total_success} | Failed: {total_failed}")

In [0]:
catalog = "workspace"

# Get all Delta tables
tables = spark.sql(f"""
    SELECT table_schema, table_name
    FROM {catalog}.information_schema.tables
    WHERE data_source_format = 'DELTA'
    AND table_schema != 'information_schema'
""").collect()

print(f"{'Schema':<30} {'Table':<40} {'Log Retention':<25} {'File Retention'}")
print("="*120)

for row in tables:
    fqn = f"`{catalog}`.`{row.table_schema}`.`{row.table_name}`"
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {fqn}").collect()
        props_dict = {r["key"]: r["value"] for r in props}

        log_ret  = props_dict.get("delta.logRetentionDuration", "not set (default 7 days)")
        file_ret = props_dict.get("delta.deletedFileRetentionDuration", "not set (default 7 days)")

        print(f"{row.table_schema:<30} {row.table_name:<40} {log_ret:<25} {file_ret}")
    except Exception as e:
        print(f"{row.table_schema:<30} {row.table_name:<40} ERROR: {e}")